# Ice/Water Classification

Author: Daniel Vandevort

Date: 2025-06-19

email: vandevod@oregonstate.edu

The classification is 1 for ice if NDSI > 0.5 else 0 for no ice.

### Note regarding string names for input date files
As with most data science projects, input datafiles may have different names for input data (like the column names in a dataframe) than what may exist in your input datafiles. For names that are not generated within the code, ensure that your input data files are named to match or, perhaps more simply, change the string names in this code to match your input data. The naming convention itself has no bearing on the quality of analysis (but certainly on the readability...) and is subjective to the user. Do what makes sense to you :)

In [1]:
# imports and directories
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import matplotlib.pyplot as plt

#### DIRECTORIES ###############
# hls2_dir = r'USER_INPUT' # input
# icp_dir = r'USER_INPUT' # output

## CLASSIFICATION

In [ ]:
# Get total number of rows across all files (for progress bar)
total_rows = 0

hls2_files = [f for f in os.listdir(hls2_dir) if f.endswith(".csv")] # or other file extension
for f in hls2_files:
    path = os.path.join(hls2_dir, f)
    try:
        total_rows += sum(1 for _ in open(path)) - 1  # get rows excluding header
    except Exception as e:
        print(f"Error calculating rows for file {f}: {e}")
# Start tqdm progress bar - handy for progress tracking but not necessary.
with tqdm(total=total_rows, desc='Classifying all pixels', unit='rows') as pbar:
    for f in hls2_files:
        path = os.path.join(hls2_dir, f)
        try:
            df = pd.read_csv(path)
            df['ice_class'] = None  # Initialize the new column
            for index, row in df.iterrows():
                # NIR_S = near-infrared from Sentinel-2, NIR_L = near-infrared from Landsat.
                df.at[index, 'ice_class'] = 1 if row['NDSI'] >= 0.4 and (row['NIR_S'] > 0.11 or row['NIR_L'] > 0.11) else 0 # 1 = ice, 0 = water
                pbar.update(1)  # Update progress bar for each processed row
            df.to_csv(path, index=False)  # Overwrite existing file with classification
        except Exception as e:
            print(f"Failed to process file {f}: {e}")

## ICP (Ice coverage percentage) Calculation.
Note: this method was developed for an earlier version of this project. While it is not used directly in the final analysis, it is retained here for reference and potential future use.

In [ ]:
# Define the directory to store the ICP results
def calculate_icp(df):
    grouped = df.groupby(['lake_id', 'date'])
    results = []
    for (lake_id, date), g in grouped:
        total_pixels = len(g)
        ice_pixels = g['ice_class'].sum()  # 1 for ice, 0 for water
        if total_pixels > 0:
            icp = (ice_pixels / total_pixels) * 100
        else:
            icp = 0.0
        # Classify ice cover
        if icp <= 25:
            ice_class = 'open'
        elif 25 < icp < 75:
            ice_class = 'partial'
        else:
            ice_class = 'full'

        results.append({
            'lake_id': lake_id,
            'date': date,
            'hls2_icp_%': icp,
            'ice_class': ice_class
        })
    return pd.DataFrame(results)
print('Beginning ICP classification...')
for f in tqdm([f for f in os.listdir(hls2_dir)], desc='Calculating ICP...', unit='file'): # Processes each classified CSV file
    classified_df = pd.read_csv(os.path.join(hls2_dir, f))
    icp_df = calculate_icp(classified_df) # calc ICP
    for lake_id in icp_df['lake_id'].unique():
        lake_df = icp_df[icp_df['lake_id'] == lake_id]
        # Save the ICP results to a new CSV file for each lake
        output_file = os.path.join(icp_dir, f'lake_{lake_id}.csv')
        lake_df.to_csv(output_file, index=False) # every lake gets a file.
print("ICP classification complete.")


## NDSI classification sensitivity analysis

In [ ]:
# Load metadata
class_dir = r'path/to/ndsi/files'  # USER INPUT
metadata_df = pd.read_csv(r'path/to/metadata/file', # IF Separate, else ignore.
                          names=['lake_id', 'latitude', 'elevation_m', 'surface_area_km2', 'state'], # change names as needed
                          header=0, on_bad_lines= "skip")

In [ ]:
# Filter to lakes that have NDSI files. Might be necessary if some data files are corrupted or missing.
def get_available_lakes(input_df, input_dir):
    """Filter metadata to only lakes with existing NDSI CSV files."""
    available_ids = []
    for lake_id in input_df['lake_id']:
        filepath = os.path.join(input_dir, f'lake_id_{lake_id}.csv')
        if os.path.exists(filepath):
            available_ids.append(lake_id)
        else:
            print(f"Lake ID {lake_id} does not have an NDSI file and will be excluded.")
    return input_df[input_df['lake_id'].isin(available_ids)].copy()

metadata_df = get_available_lakes(metadata_df, class_dir)

In [ ]:
def stratified_sample(metadata_df, n_per_stratum=4, seed=42): # random seed for reproducibility
    """Select lakes using stratified sampling across lat, area, elevation."""
    np.random.seed(seed)
    # Create bins
    metadata_df['lat_bin'] = pd.cut(metadata_df['latitude'], bins=3, labels=['low', 'mid', 'high'])
    metadata_df['area_bin'] = pd.cut(metadata_df['surface_area_km2'],
                                      bins=[0.01, 0.1, 0.5, np.inf], # manually set b/c of skewed distribution. change as needed
                                      labels=['small', 'medium', 'large'])
    metadata_df['elev_bin'] = pd.cut(metadata_df['elevation_m'],
                                      bins=[500, 1000, 1800, np.inf], # manually set to capture ranges of interest. change as needed.
                                      labels=['low', 'mid', 'high'])

    # Sample from each stratum
    sampled = metadata_df.groupby(['lat_bin', 'area_bin', 'elev_bin'], observed=True).apply(
        lambda x: x.sample(min(len(x), n_per_stratum), random_state=seed)
    ).reset_index(drop=True)

    print(f"Selected {len(sampled)} lakes from {metadata_df.groupby(['lat_bin', 'area_bin', 'elev_bin'], observed=True).ngroups} strata")
    return sampled

In [ ]:
sampled_lakes = stratified_sample(metadata_df)
selected_ids = sampled_lakes['lake_id'].tolist()

In [ ]:
def sensitivity_analysis(hls2_dir, lake_ids, thresholds=[0.3, 0.4, 0.5, 0.6, 0.7]):
    """Run sensitivity analysis across NDSI thresholds for selected lakes."""
    results = []

    for lake_id in tqdm(lake_ids, desc='Processing lakes'):
        filepath = os.path.join(hls2_dir, f'lake_id_{lake_id}.csv')
        if not os.path.exists(filepath):
            continue

        df = pd.read_csv(filepath)
        for threshold in thresholds:
            df['ice_pixel'] = ((df['NDSI'] >= threshold) &
                               ((df['NIR_S'] > 0.11) | (df['NIR_L'] > 0.11))).astype(int) #  1 for ice, 0 for water same as before

            grouped = df.groupby('date')['ice_pixel'].agg(['sum', 'count'])
            grouped['ice_pct'] = (grouped['sum'] / grouped['count']) * 100
            grouped['binary_class'] = (grouped['ice_pct'] >= 25).map(
                {True: 'ice_present', False: 'open_water'}) # 25 percent threshold.

            for date, row in grouped.iterrows():
                results.append({
                    'lake_id': lake_id,
                    'date': date,
                    'ndsi_threshold': threshold,
                    'ice_pct': row['ice_pct'],
                    'binary_class': row['binary_class']
                })

    return pd.DataFrame(results)

In [ ]:
# Execute analysis
sensitivity_df = sensitivity_analysis(class_dir, selected_ids)

In [ ]:
# Find unstable classifications
pivot_df = sensitivity_df.pivot_table(index=['lake_id', 'date'],
                                       columns='ndsi_threshold',
                                       values='binary_class',
                                       aggfunc='first')
pivot_df['flips'] = (pivot_df.ne(pivot_df.shift(axis=1))).sum(axis=1) - 1
print(f"Observations with classification changes: {(pivot_df['flips'] > 0).sum()}")

In [ ]:
# Show how classifications change between adjacent thresholds
pivot_df = sensitivity_df.pivot_table(index=['lake_id', 'date'],
                                       columns='ndsi_threshold',
                                       values='binary_class',
                                       aggfunc='first')

fig, ax = plt.subplots(figsize=(10, 6))
transition_counts = pivot_df.apply(lambda col: col.value_counts()).T.fillna(0)
transition_counts.plot(kind='bar', stacked=True, ax=ax, color=['darkblue', 'steelblue'])
ax.set_xlabel('NDSI Threshold', fontsize=20)
ax.set_ylabel('Number of classifications', fontsize=20)
ax.tick_params(axis='both', which='major', labelsize=16, direction = 'inout', length = 5)
ax.tick_params(axis='x', which='major', rotation=0)
ax.legend(title='Classification', fontsize = 16)
plt.tight_layout()
plt.grid(True, axis='y')
plt.show()

## NDSI+NIR classification performance.
100 images manually checked for ice/water classification.

In [5]:
# --- Load the validated data ---
df_val = pd.read_csv(r"path/to/validated/data.csv")  # USER INPUT
# --- Filter: remove uninterpretable images ---
df_clean = df_val[
    (df_val["Class Correct?"] != 2)
].copy()
print(f"Samples used for evaluation: {len(df_clean)}")

Samples used for evaluation: 95


In [6]:
# see validation data .xlsx file.
y_pred = df_clean["ice_binary"]
def infer_true_class(row):
    if row["Class Correct?"] == 1:
        return row["ice_binary"]
    elif row['ice_binary'] == 1 and row["Class Correct?"] == 0:
        return 0  # misclassified as ice, true class is water
    elif row['ice_binary'] == 0 and row["Class Correct?"] == 0:
        return 1  # misclassified as water, true class is ice
    else:
        return np.nan  # unknown or unhandled case; shouldn't happen

df_clean["true_class"] = df_clean.apply(infer_true_class, axis=1)
y_true = df_clean["true_class"]

In [10]:
# --- Reporting exclusions ---
n_cloudy = (df_val["Cloudy?"] == 1).sum()
n_unknown = (df_val["Class Correct?"] == 2).sum()
print(f" Cloudy:  {n_cloudy}")
print(f"Excluded: {n_unknown}")
print(f"Usable samples:     {len(df_clean)}")


 Cloudy:  12
Excluded: 5
Usable samples:     95


In [21]:
labels = sorted(y_true.unique())

# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred, labels=labels)
print(cm)

# --- Core Metrics ---
print("=== Classification Metrics ===")
print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
print(f"Precision: {precision_score(y_true, y_pred, average='binary', zero_division='warn'):.3f}")
print(f"Recall:    {recall_score(y_true, y_pred, average='binary', zero_division='warn'):.3f}")
print(f"F1 Score:  {f1_score(y_true, y_pred, average='binary', zero_division='warn'):.3f}")

# --- Full per-class breakdown ---
print("\n=== Per-Class Report ===")
print(classification_report(y_true, y_pred, labels=labels, zero_division=0))


[[55  1]
 [ 1 38]]
=== Classification Metrics ===
Accuracy:  0.979
Precision: 0.974
Recall:    0.974
F1 Score:  0.974

=== Per-Class Report ===
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        56
           1       0.97      0.97      0.97        39

    accuracy                           0.98        95
   macro avg       0.98      0.98      0.98        95
weighted avg       0.98      0.98      0.98        95

